In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

# make the function to predict stock price for chosen number of days using chosen amount of data
def run_fft(num_days_predict, num_days_data):
    
    # load the stock data from a CSV file
    stock_data = pd.read_csv(r'C:\Users\cl31h\Downloads\AAPL.csv')
    stock_data.index = stock_data.Date
    # extract the close prices from the stock data
    close_price = stock_data['Adj Close'].values
    random_date = random.randint(500, len(close_price) - 10) # making sure you have enough data to make the prediction and enough comparison data
          
    # create data frame & plot original data
    odf = stock_data[random_date - num_days_data:random_date][['Adj Close']]
    cdf = close_price[random_date - 1:random_date + num_days_predict]
  
    # apply the FFT on the close prices and calculate superposition values
    odf['delta'] = np.append(np.array([0]), np.diff(odf['Adj Close'].values))
    sp = np.fft.fft(odf['delta'].values)
    odf['theta'] = np.arctan(sp.imag / sp.real)
    numValues = len(odf)
    numValuesHalf = numValues / 2
    odf['amplitude'] = np.sqrt(sp.real ** 2 + sp.imag ** 2) / numValuesHalf
    odf['freq'] = np.fft.fftfreq(sp.size, d=1)

    std_values = []
    rmse_values = []
          
    for i in np.linspace(0, 2, 20):
        std_values.append(i)
        rmse_values.append(std_filter(i, odf))

    idx = np.array(rmse_values).argmin()
    minSTD = std_values[idx]

    # getting dominant values based on std_value
    meanAmp = odf['amplitude'].mean()
    stdAmp = odf['amplitude'].std()
    dominantAmpCheck = odf['amplitude'] > (minSTD * stdAmp + meanAmp)
    positiveFreqCheck = odf['freq'] > 0
    dominantAmp = odf[dominantAmpCheck & positiveFreqCheck]['amplitude']
    dominantFreq = odf[dominantAmpCheck & positiveFreqCheck]['freq']
    dominantTheta = odf[dominantAmpCheck & positiveFreqCheck]['theta']

    # calculating regression delta
    regressionDelta = 0
    for n in range(len(dominantTheta)):
        shift = dominantTheta[n]
        regressionDelta += dominantAmp[n] * np.cos(n * np.array(range(len(odf) + num_days_predict)) + shift)
    
    predict = regressionDelta[- num_days_predict - 1:]
    predict[0] = 0
          
    # converting delta time to time at start value of real data
    startValue = odf['Adj Close'][num_days_data - 1]
    regression = startValue + np.cumsum(predict)
    
    if regression[-1] > startValue and cdf[-1] > startValue:
          return 1
          
    elif regression[-1] < startValue and cdf[-1] < startValue:
          return 1
          
    else:
          return 0

              

# define std_filter to minimise RMSE      
def std_filter(std_value, df):
    # getting dominant values based on std_value
    meanAmp = df['amplitude'].mean()
    stdAmp = df['amplitude'].std()
    dominantAmpCheck = df['amplitude'] > (std_value * stdAmp + meanAmp)
    positiveFreqCheck = df['freq'] > 0
    dominantAmp = df[dominantAmpCheck & positiveFreqCheck]['amplitude']
    dominantFreq = df[dominantAmpCheck & positiveFreqCheck]['freq']
    dominantTheta = df[dominantAmpCheck & positiveFreqCheck]['theta']

    # calculating regression delta
    regressionDelta = 0
    for n in range(len(dominantTheta)):
        shift = dominantTheta[n]
        regressionDelta += dominantAmp[n] * np.cos(n * np.array(range(len(df))) + shift)

    # converting delta time to time at start value of real data
    startValue = df['Adj Close'][0]
    regression = startValue - np.cumsum(regressionDelta)

    # calculating RMSE
    rmse = np.sqrt(np.mean((df['Adj Close'].values - regression) ** 2))

    if np.isnan(rmse):
        rmse = 10000000000000

    return rmse

In [ ]:
run_fft(1,250)

In [ ]:
# test for accuracy of function
test = []
for j in range(100):
    test.append(run_fft(1,250))
sum(test)

In [ ]:
    # plot the predicted values against the real values
    plt.figure(figsize=(15, 5))
    plt.plot(cdf)
    plt.plot(regression)
    plt.legend(['Real', 'Predicted'])
    plt.axhline(y=startValue, color='r', linestyle='-') # add a horizontal line to make it clear if successful
    plt.ylabel('Stock Price [$]')
    plt.title('Apple Stock Price Prediction', fontsize=15)
    plt.savefig(r'C:\Users\cl31h\Downloads\Apple Prediction.png', format='png')